# T1.L1 — Форма і принципи представлення математичних моделей

Мета notebook: пройти повний шлях від математичної залежності до відтворюваного обчислювального експерименту.

## 1. Постановка

Маємо початковий запас `S0`, який витрачається зі сталою інтенсивністю `rate`.

Математична модель: `S(t) = S0 - rate*t`. Для фізично змістовної версії застосовуємо `S(t) >= 0`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
from model import resource, depletion_time, simulate, compare_rates

## 2. Базовий сценарій

In [ ]:
S0 = 120
rate = 8
times = range(0, 21)
base = simulate(times, s0=S0, rate=rate)
base.head()

In [ ]:
print('Теоретичний час вичерпання:', depletion_time(S0, rate))
print('S(0) =', float(resource([0], S0, rate)[0]))
print('S(10) =', float(resource([10], S0, rate)[0]))

### Sanity check

Аналітично: `120 / 8 = 15`. Якщо код дає інший час вичерпання — програмна реалізація не відповідає математичній моделі.

In [ ]:
assert depletion_time(120, 8) == 15
assert float(resource([0], 120, 8)[0]) == 120
assert float(resource([10], 120, 8)[0]) == 40

## 3. Графічне представлення

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(base['time'], base['resource'], marker='o')
ax.set_title('Базовий сценарій зміни ресурсу')
ax.set_xlabel('Час, умовні одиниці')
ax.set_ylabel('Залишок ресурсу, умовні одиниці')
ax.grid(True, alpha=0.3)
plt.show()

## 4. Математично коректно ≠ змістовно коректно

Вимкнемо фізичне обмеження і подивимося на великий `t`.

In [ ]:
print('Без обмеження:', float(resource([30], 100, 5, clamp_zero=False)[0]))
print('З обмеженням:', float(resource([30], 100, 5, clamp_zero=True)[0]))

Від’ємний запас є наслідком продовження лінійної формули за межі змістовної області. Це демонструє, чому математична модель повинна містити не лише формулу, але й припущення та обмеження.

## 5. Змінений сценарій

In [ ]:
economy = simulate(times, s0=120, rate=5)
comparison = pd.concat([base.assign(scenario='base'), economy.assign(scenario='economy')], ignore_index=True)
fig, ax = plt.subplots(figsize=(8, 5))
for name, group in comparison.groupby('scenario'):
    ax.plot(group['time'], group['resource'], marker='o', label=name)
ax.set_title('Порівняння сценаріїв')
ax.set_xlabel('Час, умовні одиниці')
ax.set_ylabel('Залишок ресурсу, умовні одиниці')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
print('Base depletion:', depletion_time(120, 8))
print('Economy depletion:', depletion_time(120, 5))

## 6. Аналіз чутливості

In [ ]:
sensitivity = compare_rates(times, s0=120, rates=[4, 6, 8, 10, 12])
fig, ax = plt.subplots(figsize=(8, 5))
for rate_value, group in sensitivity.groupby('scenario_rate'):
    ax.plot(group['time'], group['resource'], label=f'rate={rate_value:g}')
ax.set_title('Чутливість моделі до інтенсивності витрачання')
ax.set_xlabel('Час, умовні одиниці')
ax.set_ylabel('Залишок ресурсу, умовні одиниці')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 7. Інтерпретація

1. Що саме змінилося між сценаріями — модель чи параметр?
2. Чому зменшення `rate` збільшує час до вичерпання?
3. Що цей експеримент не дозволяє стверджувати?
4. Які реальні процеси вимагали б нелінійної, динамічної або стохастичної моделі?

## 8. Research transfer

Сформулюйте аналогічну структуру для власної дисертаційної проблеми: `Problem → variables → parameters → relation → assumptions → experiment → verification → interpretation`.